In [2]:
import sys
sys.path.append('..')  # or the absolute path to your project root

In [ ]:
"""
AHC (Agglomerative Hierarchical Clustering) — Drop-in replacement for HDBSCAN notebook
========================================================================================
Place AHC.py in src/models/ and visualize_ahc.py in src/evaluation/
Then run this notebook as-is.
"""

import pandas as pd

# ── 1. Load Data ──────────────────────────────────────────────────────────────
df_base = pd.read_csv("../data/users_fingerprint_norm.csv").set_index("user_id").drop(columns=["Unnamed: 0"])
df_1a = pd.read_csv("../data/user_embedding_svd_ability_seen_beh.csv").set_index("user_id")
df_1b = pd.read_csv("../data/user_embedding_svd_ability_seen_beh_B.csv").set_index("user_id")
df_2a = pd.read_csv("../data/user_embedding_svd512_ae128_plus_beh.csv").set_index("user_id")
df_2b = pd.read_csv("../data/user_embedding_svd512_ae128_plus_beh_B.csv").set_index("user_id")
df_3  = pd.read_csv("../data/user_fingerprint_B_lex_clusters_scaled.csv").set_index("user_id").drop(columns=["Unnamed: 0"])

print(df_base.shape)
print(df_1a.shape)
print(df_1b.shape)
print(df_2a.shape)
print(df_2b.shape)
print(df_3.shape)

# ── 2. Hopkins Statistic (same as HDBSCAN) ───────────────────────────────────
from src.evaluation.metrics import hopkins_statistic

for df in [df_base, df_1a, df_1b, df_2a, df_2b, df_3]:
    score = hopkins_statistic(df.values)
    print(f"Hopkins Statistic: {score:.4f}")

# ── 3. Run AHC Pipeline ──────────────────────────────────────────────────────
from src.models.ahc import run_ahc_pipeline, save_clustering_results, get_all_cluster_importances
from src.evaluation.visualize_ahc import visualize_ahc_with_labels, plot_cluster_top_features_boxplot, plot_cluster_top_features_radar

datasets = {
    "baseline": df_base,
    "svd": df_1a,
    "svd_B": df_1b,
    "svd512_ae128": df_2a,
    "svd512_ae128_B": df_2b,
    "lex_clusters_B": df_3,
}

for name, df in datasets.items():
    print(f"\n--- Processing: {name} ---")

    processed_df, model, Z = run_ahc_pipeline(
        df.reset_index(),
        n_components=min(df.shape[1], 200),
        n_clusters=10,          # <-- Tune this (or use distance_threshold instead)
        linkage_method='ward',  # 'ward', 'complete', 'average', 'single'
    )

    save_clustering_results(processed_df, model, Z, name, "../data/tmp/ahc")

    feature_cols = [c for c in processed_df.columns if c not in ['user_id', 'cluster_label']]
    importance_dict = get_all_cluster_importances(processed_df, feature_cols)

    visualize_ahc_with_labels(processed_df, Z, name, "../data/tmp/ahc/plots")

# ── 4. Inspect Individual Clusters ───────────────────────────────────────────
# Example: inspect cluster 0 from the last processed dataset
plot_cluster_top_features_radar(processed_df, 0, importance_dict)
plot_cluster_top_features_boxplot(processed_df, 0, importance_dict)

(2709, 5633)
(2709, 99)
(2709, 99)
(2709, 131)
(2709, 131)
(2709, 165)
Hopkins Statistic: 0.9739
Hopkins Statistic: 0.8658
Hopkins Statistic: 0.8567
Hopkins Statistic: 0.7533
Hopkins Statistic: 0.7684
Hopkins Statistic: 0.9670


ModuleNotFoundError: No module named 'src.models.AHC'

In [5]:
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 19.8 MB/s  0:00:00m0:00:0100:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
